In [ ]:
# !pip install --upgrade transformers datasets evaluate

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    matthews_corrcoef,
    balanced_accuracy_score,
)
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from datasets import (
    load_dataset, 
    ClassLabel, 
    DatasetDict,
)
import evaluate

/usr/local/lib/python3.10/dist-packages/torchvision/datapoints/__init__.py:14: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/usr/local/lib/python3.10/dist-packages/torchvision/transforms/v2/__init__.py:64: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https:/

In [38]:
MODEL_NAME = "distilbert-base-uncased"

TEST_SIZE = 0.2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 2
BATCH_SIZE = 16

DATA_PATH = "data/sentence_sets_trimmed.csv"
DATA_ENCODING = "ISO-8859-1"
# LABELS = {"female": 0, "male": 1}

LABEL_COLUMN = "applicant_gender"
TEXT_COLUMN = "s1_s2"

STRATIFY_ENABLED = True
DEGENDER_ENABLED = False
BALANCED_WEIGHTS_ENABLED = True

MAX_LENGTH = 512

OUTPUT_NAME = f"{MODEL_NAME}-finetuned-nlp-letters-{TEXT_COLUMN}"
OUTPUT_NAME += "-degendered" if DEGENDER_ENABLED else ""
OUTPUT_NAME += "-stratified-" if STRATIFY_ENABLED else ""
OUTPUT_NAME += "-balanced" if BALANCED_WEIGHTS_ENABLED else ""

In [39]:
class LettersBERTModule(nn.Module):
    def __init__(self, model_name=MODEL_NAME, class_weights=None, cls_or_mean="cls"):
        super().__init__()

        self.config = AutoConfig.from_pretrained(model_name)
        self.config.num_labels = num_labels
        self.config.class_weights = class_weights
        self.config.cls_or_mean = cls_or_mean

        # Layers
        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        self.classifier = nn.Linear(self.config.hidden_size, self.config.num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)

        # Either use [CLS] token or mean pooling
        if self.config.cls_or_mean == "mean":
            output = torch.mean(outputs.last_hidden_state, dim=1)
        else:
            output = outputs.last_hidden_state[:, 0, :]

        # Logits and loss
        logits = self.classifier(output)
        loss = None

        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.config.class_weights)
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}

In [40]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [41]:
def degender(text: str) -> str:
    pronouns = {
        " mr ": " mx ",
        " mrs ": " mx ",
        " ms ": " mx ",
        " miss ": " mx ",
        " mister ": " mx ",
    }
    nouns = {
        " man ": " person ",
        " men ": " persons ",
        " woman ": " person ",
        " women ": " persons ",
        " man's ": " person's ",
        " men's ": " person's ",
        " woman's ": " person's ",
        " women's ": " person's ",
        " gentleman ": " person ",
        " lady ": " person ",
        " gentleman's ": " person's ",
        " lady's ": " person's ",
    }

    text = text.lower()

    for old, new in pronouns.items():
        text = text.replace(old, new)

    for old, new in nouns.items():
        text = text.replace(old, new)

    return text

In [42]:
def preprocess(data):
    texts = data[TEXT_COLUMN]

    if DEGENDER_ENABLED:
        texts = [degender(t) for t in texts]

    tokenized = tokenizer(
        texts, 
        truncation=True, 
        padding=True
    )

#     labels = [LABELS[label_str] for label_str in data[LABEL_COLUMN]]
    tokenized["labels"] = data[LABEL_COLUMN]

    return tokenized

In [43]:
# Load the data
dataset = load_dataset("csv", data_files=DATA_PATH, encoding=DATA_ENCODING)

# Recast labels as class features
unique_labels = dataset["train"].unique(LABEL_COLUMN)

features = dataset["train"].features
features[LABEL_COLUMN] = ClassLabel(names=unique_labels)

dataset = dataset.cast(features)

dataset = dataset["train"].train_test_split(
    test_size=TEST_SIZE,
    stratify_by_column=LABEL_COLUMN if STRATIFY_ENABLED else None,
    seed=100,
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [54]:
print(dataset["train"][TEXT_COLUMN][:10])

["FIRST_NAME MIDDLE_NAME paz de araujo   a fourth   year medical student at the university of colorado health sciences center   currently applying for a position in your anesthesiology residency program  * I had the opportunity to work with FIRST_NAME during clinical clerkships at children's hospital of colorado in both his third and fourth year of medical school  * FIRST_NAME excelled in all phases of his clinical responsibilities during his pediatric anesthesiology clerkship  * I often found FIRST_NAME remaining in the or later than usual to observe and participate in interesting cases taking place later in the FIRST_NAME  * airway and intravenous access skills were outstanding and significantly above the standard expected for FIRST_NAME's current level of training  * FIRST_NAME displayed a comfort and keen ability to interact and connect with pediatric patients and their families  * FIRST_NAME's commitment to public health and research are evident with his participation in the sbirt

In [44]:
train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

Map:   0%|          | 0/657 [00:00<?, ? examples/s]

In [45]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")
confusion_metric = evaluate.load("confusion_matrix")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=preds, references=labels)
    precision = precision_metric.compute(predictions=preds, references=labels, average="macro")
    recall = recall_metric.compute(predictions=preds, references=labels, average="macro")
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")

    mcc = matthews_corrcoef(labels, preds)
    bal_acc = balanced_accuracy_score(labels, preds)
    
    cr = classification_report(labels, preds, target_names=train_dataset.features[LABEL_COLUMN].names)
    cm = confusion_metric.compute(predictions=preds, references=labels)

    print("Confusion Matrix:\n", cm["confusion_matrix"])
    print("Classification Report:\n", cr)
    print("MCC:", mcc)
    print("Balanced Accuracy:", bal_acc)

    return {
        "accuracy": accuracy["accuracy"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "f1": f1["f1"],
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
    }

In [46]:
# Class weight balancing
class_weights = torch.tensor(
    compute_class_weight(
        "balanced", 
        classes=np.unique(train_dataset[LABEL_COLUMN]), 
        y=train_dataset[LABEL_COLUMN]
    ), dtype=torch.float
)

In [47]:
# model = LettersBERTModule(model_name=MODEL_NAME, num_labels=len(LABELS), class_weights=class_weights)

In [48]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=len(train_dataset.features[LABEL_COLUMN].names)
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [49]:
# Training arguments
training_args = TrainingArguments(
    output_dir=f"./{OUTPUT_NAME}",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

/home/hice1/mwesley32/.local/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_3220858/1892133964.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [50]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mcc,Balanced Accuracy
1,0.005800,0.011351,0.998478,0.998938,0.997326,0.998128,0.996263,0.997326
2,0.003300,0.011142,0.998478,0.998938,0.997326,0.998128,0.996263,0.997326


Confusion Matrix:
 [[470   0]
 [  1 186]]
Classification Report:
               precision    recall  f1-score   support

        male       1.00      1.00      1.00       470
      female       1.00      0.99      1.00       187

    accuracy                           1.00       657
   macro avg       1.00      1.00      1.00       657
weighted avg       1.00      1.00      1.00       657

MCC: 0.9962633275738177
Balanced Accuracy: 0.9973262032085561
Confusion Matrix:
 [[470   0]
 [  1 186]]
Classification Report:
               precision    recall  f1-score   support

        male       1.00      1.00      1.00       470
      female       1.00      0.99      1.00       187

    accuracy                           1.00       657
   macro avg       1.00      1.00      1.00       657
weighted avg       1.00      1.00      1.00       657

MCC: 0.9962633275738177
Balanced Accuracy: 0.9973262032085561


TrainOutput(global_step=330, training_loss=0.03964308888623209, metrics={'train_runtime': 100.7245, 'train_samples_per_second': 52.182, 'train_steps_per_second': 3.276, 'total_flos': 696248647335936.0, 'train_loss': 0.03964308888623209, 'epoch': 2.0})

In [51]:
trainer.evaluate()

Confusion Matrix:
 [[470   0]
 [  1 186]]
Classification Report:
               precision    recall  f1-score   support

        male       1.00      1.00      1.00       470
      female       1.00      0.99      1.00       187

    accuracy                           1.00       657
   macro avg       1.00      1.00      1.00       657
weighted avg       1.00      1.00      1.00       657

MCC: 0.9962633275738177
Balanced Accuracy: 0.9973262032085561


{'eval_loss': 0.011351289227604866,
 'eval_accuracy': 0.9984779299847792,
 'eval_precision': 0.9989384288747346,
 'eval_recall': 0.9973262032085561,
 'eval_f1': 0.9981281677982182,
 'eval_mcc': 0.9962633275738177,
 'eval_balanced_accuracy': 0.9973262032085561,
 'eval_runtime': 3.8779,
 'eval_samples_per_second': 169.423,
 'eval_steps_per_second': 10.831,
 'epoch': 2.0}